# Invoice OCR — Molab
Chạy các cell từ trên xuống trên **Linux / NVIDIA GPU / Python >= 3.12**.
Notebook chỉ cấu hình và gọi **`scripts/molab_benchmark.py` từ GitHub**.
Push script lên GitHub trước. Token nhập ẩn; mặc định chấm reviewed `valid`.


In [ ]:
REPO = "ndkhanhus27/invoice-ocr"
REF = "main"
GITHUB_PRIVATE = False  # True nếu repo code private.
CONFIG = {
    "github_repo": REPO, "github_ref": REF,
    "dataset": "ndkhanhus27/invoice-ocr-data", "dataset_ref": "main",
    "split": "valid", "limit": 3,  # 0 = toàn bộ reviewed, 3 = chạy thử.
    "model": "gemma4:31b-it-q8_0", "ollama_version": "latest",
    "num_ctx": 16384, "max_tokens": 8192, "max_edge": 1600, "timeout": 1800,
}


## Lấy script từ GitHub


In [ ]:
import getpass, importlib.util, json, os, tempfile
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.parse import quote

github_token = os.environ.get("GITHUB_TOKEN", "")
if GITHUB_PRIVATE and not github_token:
    github_token = getpass.getpass("GitHub token (ẩn): ")
headers = {"User-Agent": "invoice-ocr-molab"}
if github_token:
    headers["Authorization"] = "Bearer " + github_token
with urlopen(Request(f"https://api.github.com/repos/{REPO}/commits/{quote(REF, safe='')}", headers=headers), timeout=60) as response:
    commit = json.load(response)["sha"]
with urlopen(Request(f"https://raw.githubusercontent.com/{REPO}/{commit}/scripts/molab_benchmark.py", headers=headers), timeout=60) as response:
    script = Path(tempfile.mkdtemp(prefix="invoice-code-")) / "molab_benchmark.py"
    script.write_bytes(response.read())
spec = importlib.util.spec_from_file_location("molab_benchmark", script)
benchmark = importlib.util.module_from_spec(spec)
spec.loader.exec_module(benchmark)
CONFIG["github_ref"] = commit
print("Đã lấy script GitHub:", commit, flush=True)


## Cài thư viện


In [ ]:
benchmark.install_dependencies()


## Chạy model và chấm điểm
Cell này bắt đầu tải full data/model. Log và lỗi hiện trực tiếp.


In [ ]:
result = benchmark.run_from_notebook(CONFIG, github_token=github_token)


## Điểm và tải ZIP


In [ ]:
benchmark.show_results(result)
